In [3]:
# keep this cell collapsed to not show lol
import os
import time
import random
import colorsys
import openpyxl
import subprocess
import numpy as np
import pandas as pd
import gurobipy as gp
from openpyxl.styles import PatternFill
from numpy import rot90, fliplr


def wait_for_file_change(file_path, timeout=None):
    initial_time = os.path.getmtime(file_path)
    while True:
        current_time = os.path.getmtime(file_path)
        if current_time != initial_time:
            print(f"File '{file_path}' has been modified.")
            time.sleep(0.2)
            return True
        
        if timeout is not None and time.time() - initial_time >= timeout:
            print("Timeout reached.")
            return False
        
        time.sleep(0.05)

pieces = []
piece_single = np.array([
    [1]
])
piece_double = np.array([
    [1, 1],
    [1, 1]
])
pieces.extend([piece_single, piece_double])

pieces.extend([np.repeat(piece_single, 2, axis=1), np.repeat(piece_single, 3, axis=1), np.repeat(piece_single, 4, axis=1)])
pieces.extend([rot90(np.repeat(piece_single, 2, axis=1)), rot90(np.repeat(piece_single, 3, axis=1)), rot90(np.repeat(piece_single, 4, axis=1))])

piece_l = np.array([
    [1, 0],
    [1, 1]
])
pieces.extend([piece_l, rot90(piece_l), rot90(piece_l, 3), rot90(piece_l, 2)])

piece_L = np.array([
    [1, 0, 0],
    [1, 1, 1]
])
pieces.extend([piece_L, fliplr(piece_L), rot90(fliplr(piece_L), 2), rot90(piece_L, 2)])
pieces.extend([rot90(fliplr(piece_L), 3), rot90(piece_L), rot90(piece_L, 3), rot90(fliplr(piece_L))])

piece_S = np.array([
    [0, 1, 1],
    [1, 1, 0]
])
pieces.extend([piece_S, fliplr(piece_S), rot90(piece_S, 1), rot90(fliplr(piece_S))])

piece_T = np.array([
    [0, 1, 0],
    [1, 1, 1]
])
pieces.extend([piece_T, rot90(piece_T, 2), rot90(piece_T, 3), rot90(piece_T, 1)])

def secret_tweaks(model: gp.Model):
    model.setParam('OutputFlag', 0)
    model.setParam('Presolve', 0)
    model.setParam('PoolSolutions', 1) 
    model.setParam('MIPFocus', 1)  # Focus on finding feasible solutions quickly
    model.setParam('TimeLimit', 3600)  # Solve for a maximum of 3600 seconds
    model.setParam('MIPGap', 0.01)  # Set a 1% optimality gap tolerance
    model.setParam('Threads', 16)  # Use 16 threads for parallel solving
    model.setParam('OutputFlag', 1)

def decode_board(encoded_board_df: pd.DataFrame):
    column_width = int(encoded_board_df.iloc[0, 0])
    encoded_board = encoded_board_df.iloc[1:,0].fillna(0).astype(np.int64).tolist()
    board = []
    color = True  # Start with black cells
    for count in encoded_board:
        board.extend([color] * int(count))
        color = not color
    return np.array(board, dtype=np.bool_).reshape(-1, column_width)

def get_placed_pieces(placements, available_pieces):
    placed_pieces = []
    for i, piece in enumerate(available_pieces):
        for row in range(placements.shape[1]):
            for col in range(placements.shape[2]):
                if placements[i][row][col] is not None and placements[i][row][col].X:
                    placed_pieces.append((piece, (row, col)))
    return placed_pieces

In [4]:
PATH = 'fast_hands_off.xlsx'
wait_for_file_change(PATH)
solution_sheet_name = 'Solution'
input_df = pd.read_excel(PATH, sheet_name='Input', header=None)
board_df = input_df.iloc[28:].reset_index(drop=True)

In [5]:
board = decode_board(board_df)

In [6]:
counts = input_df.iloc[:len(pieces), 0].astype(int).values
available_pieces = [item for sublist in [[piece] * count for piece, count in zip(pieces, counts)] for item in sublist]

In [7]:
model = gp.Model("doodleFit")
placements = np.full((len(available_pieces), board.shape[0], board.shape[1]), None, dtype=object)
[[[placements.__setitem__((i, row, col), model.addVar(vtype=gp.GRB.BINARY, name=f'pattern_{i}_{row}_{col}')) if ~np.any((board[row:row + len(pattern), col:col + len(pattern[0])] == 1) & (pattern == 1)) else None for col in range(board.shape[1] - len(pattern[0]) + 1)] for row in range(board.shape[0] - len(pattern) + 1)] for i, pattern in enumerate(available_pieces)]
constraints = [model.addConstr(gp.quicksum([board[i][j]] + [placements[p][i - ii][j - jj] for p in range(len(available_pieces)) for ii in range(len(available_pieces[p])) for jj in range(len(available_pieces[p][0])) if available_pieces[p][ii][jj] == 1 and i - ii >= 0 and j - jj >= 0 and placements[p][i - ii][j - jj] is not None]) <= 1, f'no_overlap_{i}_{j}') for i in range(board.shape[0]) for j in range(board.shape[1])] + [model.addConstr(gp.quicksum(placements[i][row][col] for row in range(board.shape[0]) for col in range(board.shape[1]) if placements[i][row][col]) <= 1, f'one_placement_piece_{i}') for i, pattern in enumerate(available_pieces)]
# if all available_pieces have to be used below
# constraints = [model.addConstr(gp.quicksum([board[i][j]] + [placements[p][i - ii][j - jj] for p in range(len(available_pieces)) for ii in range(len(available_pieces[p])) for jj in range(len(available_pieces[p][0])) if available_pieces[p][ii][jj] == 1 and i - ii >= 0 and j - jj >= 0 and placements[p][i - ii][j - jj] is not None]) == 1, f'no_overlap_{i}_{j}') for i in range(board.shape[0]) for j in range(board.shape[1])] + [model.addConstr(gp.quicksum(placements[i][row][col] for row in range(board.shape[0]) for col in range(board.shape[1]) if placements[i][row][col]) == 1, f'one_placement_piece_{i}') for i, pattern in enumerate(available_pieces)]
pattern_counts = np.array([np.count_nonzero(pattern) for pattern in available_pieces])
model.setObjective(gp.quicksum(placements[p][i][j]*pattern_counts[p] for p in range(len(available_pieces)) for i in range(board.shape[0]) for j in range(board.shape[1]) if placements[p][i][j] is not None)+np.count_nonzero(board), gp.GRB.MAXIMIZE)
model.update()
secret_tweaks(model)
model.optimize()

Set parameter Username
Academic license - for non-commercial use only - expires 2024-12-26
Set parameter OutputFlag to value 1
Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (linux64 - "Ubuntu 22.04.3 LTS")

CPU model: 12th Gen Intel(R) Core(TM) i5-12500H, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 71 rows, 361 columns and 1153 nonzeros
Model fingerprint: 0xf614e0a1
Variable types: 0 continuous, 361 integer (361 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
Found heuristic solution: objective 36.0000000
Variable types: 0 continuous, 361 integer (361 binary)

Root relaxation: cutoff, 89 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

In [8]:
placed_pieces = get_placed_pieces(placements, available_pieces)

### Write output to excel

In [9]:
def generate_argb_colors(length):
    colors = []
    hue_values = [i / length for i in range(length)]
    random.shuffle(hue_values)
    saturation = 0.8
    value = 0.8

    for hue in hue_values:
        rgb = colorsys.hsv_to_rgb(hue, saturation, value)
        argb_color = "00{:02x}{:02x}{:02x}".format(
            int(rgb[0] * 255),
            int(rgb[1] * 255),
            int(rgb[2] * 255),
        )
        colors.append(argb_color)

    return colors

def has_neighbour_in_dir(piece, i, j, dir: str):
    try:
        if dir == "up" and i > 0:
            return piece[i - 1][j] == 1
        elif dir == "down" and i < piece.shape[0] - 1:
            return piece[i + 1][j] == 1
        elif dir == "left" and j > 0:
            return piece[i][j - 1] == 1
        elif dir == "right" and j < piece.shape[1] - 1:
            return piece[i][j + 1] == 1
        else:
            return False
    except IndexError:
        return False

def get_complementary_hex_color(hex_color):
    rgb_color = tuple(int(hex_color[i:i+2], 16) for i in (2, 4, 6))
    comp_rgb_color = tuple(255 - x for x in rgb_color)
    comp_hex_color = ''.join('{:02x}'.format(x) for x in comp_rgb_color)
    return "00"+comp_hex_color

def generate_border_style(shape, i, j):
    return openpyxl.styles.Border(
        left=(openpyxl.styles.Side(style="medium") if not has_neighbour_in_dir(shape, i, j, "left") else openpyxl.styles.Side()),
        right=(openpyxl.styles.Side(style="medium") if not has_neighbour_in_dir(shape, i, j, "right") else openpyxl.styles.Side()),
        top=(openpyxl.styles.Side(style="medium") if not has_neighbour_in_dir(shape, i, j, "up") else openpyxl.styles.Side()),
        bottom=(openpyxl.styles.Side(style="medium") if not has_neighbour_in_dir(shape, i, j, "down") else openpyxl.styles.Side())
    )

def set_cell_properties(cell, idx, color_range):
    cell.alignment = openpyxl.styles.Alignment(horizontal="center", vertical="center")
    cell.fill = PatternFill("solid", fgColor=color_range[idx])
    cell.font = openpyxl.styles.Font(color=get_complementary_hex_color(cell.fill.fgColor.rgb))

def append_border_style(existing_border, new_border):
    if not existing_border:
        return new_border
    return openpyxl.styles.Border(
        left=new_border.left if new_border.left is not None and new_border.left.style else existing_border.left,
        right=new_border.right if new_border.right is not None and new_border.right.style else existing_border.right,
        top=new_border.top if new_border.top is not None and new_border.top.style else existing_border.top,
        bottom=new_border.bottom if new_border.bottom is not None and new_border.bottom.style else existing_border.bottom
    )

In [10]:
output_book = openpyxl.load_workbook(PATH)
output_sheet = output_book[solution_sheet_name]
# clear the sheet
for row in output_sheet.iter_rows():
    for cell in row:
        cell.value = None
        cell.fill = PatternFill(fill_type="solid", fgColor="FFFFFF")
        cell.border = openpyxl.styles.Border()
color_range = generate_argb_colors(len(placed_pieces))

# place the pieces
for idx, piece in enumerate(placed_pieces):
    shape = piece[0]
    for i in range(shape.shape[0]):
        for j in range(shape.shape[1]):
            if shape[i][j]:
                cell = output_sheet.cell(piece[1][0] + 2 + i, piece[1][1] + 2 + j)
                set_cell_properties(cell, idx, color_range)
                cell.border = generate_border_style(shape, i, j)

# show the board
for y in range(board.shape[0]):
    for x in range(board.shape[1]):
        cell = output_sheet.cell(row=y + 2, column=x + 2)
        if board[y][x]:
            cell.fill = PatternFill(fill_type="solid", fgColor="000000")
        # elif the cell has no fill, fill it with white
        elif cell.fill.start_color.index == "FFFFFFFF":
            cell.fill = PatternFill(fill_type="solid", fgColor="FFFFFF")
            cell.alignment = openpyxl.styles.Alignment(horizontal="center", vertical="center")
            # fill the cell with the text "O"

# create a thick border around the board, starting at cell B2
for y in range(1, board.shape[0] + 1):
    output_sheet.cell(row=y + 1, column=2).border = append_border_style(output_sheet.cell(row=y + 1, column=2).border, openpyxl.styles.Border(left=openpyxl.styles.Side(style="thick")))
    output_sheet.cell(row=y + 1, column=board.shape[1] + 1).border = append_border_style(output_sheet.cell(row=y + 1, column=board.shape[1] + 1).border, openpyxl.styles.Border(right=openpyxl.styles.Side(style="thick")))
for x in range(1, board.shape[1] + 1):
    output_sheet.cell(row=2, column=x + 1).border = append_border_style(output_sheet.cell(row=2, column=x + 1).border, openpyxl.styles.Border(top=openpyxl.styles.Side(style="thick")))
    output_sheet.cell(row=board.shape[0] + 1, column=x + 1).border = append_border_style(output_sheet.cell(row=board.shape[0] + 1, column=x + 1).border, openpyxl.styles.Border(bottom=openpyxl.styles.Side(style="thick")))

# Save your modifications
output_book.save(PATH)

In [11]:
command = r"""/mnt/c/Windows/System32/cmd.exe /c start /mnt/c/Program\ Files/Microsoft\ Office/root/Office16/EXCEL.EXE '\\wsl.localhost\Ubuntu\home\adrian\itdc\70160513-Introduction-to-decision-making\itdm\fast_hands_off.xlsx'"""

subprocess.run(command, shell=True)

'\\wsl.localhost\Ubuntu\home\adrian\itdc\70160513-Introduction-to-decision-making\itdm'
CMD.EXE was started with the above path as the current directory.
UNC paths are not supported.  Defaulting to Windows directory.


CompletedProcess(args="/mnt/c/Windows/System32/cmd.exe /c start /mnt/c/Program\\ Files/Microsoft\\ Office/root/Office16/EXCEL.EXE '\\\\wsl.localhost\\Ubuntu\\home\\adrian\\itdc\\70160513-Introduction-to-decision-making\\itdm\\fast_hands_off.xlsx'", returncode=0)